In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="17NgTXsoTTSbhv850dK8Gm-xp_2nN6UZz", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# TP + SP Combined: Full Transformer Layer with Memory Profiling

*Notebook 3 of the Sequence Parallelism series — Vizuara GPU Programming Course*

*Estimated time: 45-60 minutes*

---

In the previous notebooks, we identified the memory waste problem (Notebook 1) and implemented the core AllGather/Reduce-Scatter primitives (Notebook 2). In this final notebook, we put it all together: a complete transformer layer that combines tensor parallelism and sequence parallelism, with full memory profiling to measure the actual savings.

We will:
1. Build a clean `TPSPTransformerLayer` class
2. Compare it against a standard TP layer
3. Profile GPU memory during forward and backward passes
4. Measure how memory scales with sequence length and TP degree

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[Open AI Teaching Assistant](https://pods.vizuara.ai/courses/gpu-programming/practice/7/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*

In [ ]:
#@title 🎧 Code Walkthrough: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import sys
import time

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"   Memory: {gpu_mem_gb:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. Running on CPU (memory profiling will be simulated).")

print(f"Python {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline

In [ ]:
#@title 🎧 Narration: Section Building Blocks
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_section_building_blocks.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 1. Building Blocks: Simulated Distributed Primitives

We reuse and extend our simulated collective operations from Notebook 2. Since we are running on a single GPU (Colab T4), we simulate $N$ GPUs by explicitly managing separate tensor buffers.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Distributed Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_simulated_distributed_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def bytes_to_mb(num_bytes):
    return num_bytes / (1024 ** 2)

def bytes_to_gb(num_bytes):
    return num_bytes / (1024 ** 3)


class SimulatedDistributed:
    """Simulates distributed collective operations for N GPUs on a single device.
    
    This class implements AllGather, Reduce-Scatter, and AllReduce as they
    would work across multiple GPUs, but operates on lists of tensors on
    a single device.
    """
    
    def __init__(self, world_size):
        self.world_size = world_size
        self.comm_bytes = 0  # track total communication
    
    def reset_comm_counter(self):
        self.comm_bytes = 0
    
    def all_gather_seq(self, chunks):
        """AllGather along sequence dim: (b, s/N, h) -> (b, s, h) on all GPUs."""
        full = torch.cat(chunks, dim=1)
        # Communication: each GPU sends s/N*h elements to N-1 others
        elem_per_chunk = chunks[0].numel()
        self.comm_bytes += 2 * (self.world_size - 1) * elem_per_chunk * chunks[0].element_size()
        return [full.clone() for _ in range(self.world_size)]
    
    def reduce_scatter_seq(self, tensors):
        """Reduce-Scatter along sequence dim: (b, s, h) partial sums -> (b, s/N, h)."""
        reduced = sum(tensors)
        chunk_size = reduced.shape[1] // self.world_size
        chunks = list(reduced.split(chunk_size, dim=1))
        # Communication: each GPU sends/receives (N-1)/N of the data
        elem_total = tensors[0].numel()
        self.comm_bytes += 2 * (self.world_size - 1) / self.world_size * elem_total * tensors[0].element_size()
        return chunks
    
    def all_reduce(self, tensors):
        """AllReduce (sum): same-shape tensors -> summed result on all GPUs."""
        reduced = sum(tensors)
        self.comm_bytes += 2 * (self.world_size - 1) / self.world_size * tensors[0].numel() * tensors[0].element_size()
        return [reduced.clone() for _ in range(self.world_size)]


dist = SimulatedDistributed(4)
print(f"Simulated distributed group: {dist.world_size} GPUs")

In [ ]:
#@title 🎧 Narration: Section Standard Tp Layer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_section_standard_tp_layer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. The Standard TP Transformer Layer (Baseline)

First, let us build a standard tensor-parallel transformer layer **without** sequence parallelism. This uses AllReduce at the exit of each sub-block, and all non-TP operations (LayerNorm, dropout) see the full activation tensor.

In [ ]:
#@title 🎧 Code Walkthrough: Standard Tp Layer Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_standard_tp_layer_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class StandardTPLayer:
    """Simulated transformer layer with standard tensor parallelism.
    
    Each GPU sees full (b, s, h) for non-TP ops and (b, s, h/N) for TP ops.
    Uses AllReduce to synchronize after each sub-block.
    """
    
    def __init__(self, hidden_dim, num_heads, tp_degree, mlp_ratio=4):
        self.h = hidden_dim
        self.N = tp_degree
        self.mlp_ratio = mlp_ratio
        
        # LayerNorms (shared across GPUs — same weights)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        
        # Column-parallel weights for attention (Q,K,V combined + output)
        self.attn_col_weights = [
            torch.randn(hidden_dim, hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.attn_row_weights = [
            torch.randn(hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
        
        # MLP weights
        self.mlp_col_weights = [
            torch.randn(hidden_dim, mlp_ratio * hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.mlp_row_weights = [
            torch.randn(mlp_ratio * hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
    
    def forward(self, inputs, dist_group):
        """Forward pass. inputs: list of N tensors, each (b, s, h)."""
        N = self.N
        stored_activations = []  # track what gets stored for backward
        
        # --- Attention sub-block ---
        residuals = [x.clone() for x in inputs]
        
        # LayerNorm on full tensor
        ln1_out = [self.ln1(inputs[r]) for r in range(N)]
        stored_activations.append(('ln1_input', inputs[0].shape, 'non-tp'))
        
        # Column-parallel (each GPU: (b,s,h) -> (b,s,h/N))
        col_out = [ln1_out[r] @ self.attn_col_weights[r] for r in range(N)]
        stored_activations.append(('attn_col_out', col_out[0].shape, 'tp'))
        
        # Simulated attention + GeLU
        activated = [F.gelu(col_out[r]) for r in range(N)]
        stored_activations.append(('attn_gelu', activated[0].shape, 'tp'))
        
        # Row-parallel (each GPU: (b,s,h/N) -> (b,s,h) partial sum)
        row_out = [activated[r] @ self.attn_row_weights[r] for r in range(N)]
        
        # AllReduce
        allreduced = dist_group.all_reduce(row_out)
        
        # Dropout + Residual on full tensor
        out1 = [F.dropout(allreduced[r], p=0.1, training=False) + residuals[r] for r in range(N)]
        stored_activations.append(('dropout1', out1[0].shape, 'non-tp'))
        stored_activations.append(('dropout1_mask', out1[0].shape, 'non-tp'))
        
        # --- MLP sub-block ---
        residuals2 = [x.clone() for x in out1]
        
        ln2_out = [self.ln2(out1[r]) for r in range(N)]
        stored_activations.append(('ln2_input', out1[0].shape, 'non-tp'))
        
        mlp_col = [ln2_out[r] @ self.mlp_col_weights[r] for r in range(N)]
        stored_activations.append(('mlp_col_out', mlp_col[0].shape, 'tp'))
        
        mlp_act = [F.gelu(mlp_col[r]) for r in range(N)]
        stored_activations.append(('mlp_gelu', mlp_act[0].shape, 'tp'))
        
        mlp_row = [mlp_act[r] @ self.mlp_row_weights[r] for r in range(N)]
        
        allreduced2 = dist_group.all_reduce(mlp_row)
        
        out2 = [F.dropout(allreduced2[r], p=0.1, training=False) + residuals2[r] for r in range(N)]
        stored_activations.append(('dropout2', out2[0].shape, 'non-tp'))
        stored_activations.append(('dropout2_mask', out2[0].shape, 'non-tp'))
        stored_activations.append(('residual', out2[0].shape, 'non-tp'))
        
        return out2, stored_activations


print("StandardTPLayer defined.")

In [ ]:
#@title 🎧 Narration: Section Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_section_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Your Turn: Build the TP+SP Transformer Layer

### TODO: Complete the TPSPLayer class

This is the main exercise. Build a transformer layer that uses:
- **Sequence parallelism** for LayerNorm, dropout, and residual (activations: $(b, s/N, h)$)
- **Tensor parallelism** for attention and MLP (activations: $(b, s, h/N)$)
- **AllGather** to transition SP -> TP
- **Reduce-Scatter** to transition TP -> SP

In [ ]:
#@title 🎧 Before You Start: Todo Tpsp Layer
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_todo_tpsp_layer.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TPSPLayer:
    """Transformer layer with combined Tensor + Sequence Parallelism.
    
    SP regions: LayerNorm, dropout, residual — activations (b, s/N, h)
    TP regions: attention, MLP — activations (b, s, h/N)
    Transitions: AllGather (SP->TP), Reduce-Scatter (TP->SP)
    """
    
    def __init__(self, hidden_dim, num_heads, tp_degree, mlp_ratio=4):
        self.h = hidden_dim
        self.N = tp_degree
        self.mlp_ratio = mlp_ratio
        
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        
        # Same weight shards as StandardTPLayer
        self.attn_col_weights = [
            torch.randn(hidden_dim, hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.attn_row_weights = [
            torch.randn(hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
        self.mlp_col_weights = [
            torch.randn(hidden_dim, mlp_ratio * hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.mlp_row_weights = [
            torch.randn(mlp_ratio * hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
    
    def forward(self, sp_inputs, dist_group):
        """Forward pass with TP + SP.
        
        Args:
            sp_inputs: list of N tensors, each (b, s/N, h) — SP layout
            dist_group: SimulatedDistributed
        
        Returns:
            sp_outputs: list of N tensors, each (b, s/N, h) — SP layout
            stored_activations: list of (name, shape, category) tuples
        """
        N = self.N
        stored_activations = []
        
        # ============ TODO ============
        # --- Attention sub-block ---
        
        # Save residual (SP layout)
        residuals = [x.clone() for x in sp_inputs]
        
        # Step 1: LayerNorm 1 — local on SP chunks (b, s/N, h)
        ln1_out = ???  # YOUR CODE: apply self.ln1 to each GPU's chunk
        stored_activations.append(('ln1_input', sp_inputs[0].shape, 'sp'))
        
        # Step 2: AllGather — (b, s/N, h) -> (b, s, h)
        full_attn = ???  # YOUR CODE: use dist_group.all_gather_seq
        
        # Step 3: Column-parallel attention — (b, s, h) -> (b, s, h/N)
        attn_col = [full_attn[r] @ self.attn_col_weights[r] for r in range(N)]
        stored_activations.append(('attn_col_out', attn_col[0].shape, 'tp'))
        
        # Step 4: Simulated attention + activation
        attn_act = [F.gelu(attn_col[r]) for r in range(N)]
        stored_activations.append(('attn_gelu', attn_act[0].shape, 'tp'))
        
        # Step 5: Row-parallel — (b, s, h/N) -> (b, s, h) partial sums
        attn_row = [attn_act[r] @ self.attn_row_weights[r] for r in range(N)]
        
        # Step 6: Reduce-Scatter — (b, s, h) partial -> (b, s/N, h)
        attn_sp = ???  # YOUR CODE: use dist_group.reduce_scatter_seq
        
        # Step 7: Dropout + Residual (SP layout)
        out1 = ???  # YOUR CODE: dropout + residual for each GPU
        stored_activations.append(('dropout1', out1[0].shape, 'sp'))
        stored_activations.append(('dropout1_mask', out1[0].shape, 'sp'))
        
        # --- MLP sub-block ---
        residuals2 = [x.clone() for x in out1]
        
        # Step 8: LayerNorm 2 — local on SP chunks
        ln2_out = ???  # YOUR CODE
        stored_activations.append(('ln2_input', out1[0].shape, 'sp'))
        
        # Step 9: AllGather — (b, s/N, h) -> (b, s, h)
        full_mlp = ???  # YOUR CODE
        
        # Step 10: Column-parallel MLP up — (b, s, h) -> (b, s, 4h/N)
        mlp_col = [full_mlp[r] @ self.mlp_col_weights[r] for r in range(N)]
        stored_activations.append(('mlp_col_out', mlp_col[0].shape, 'tp'))
        
        mlp_act = [F.gelu(mlp_col[r]) for r in range(N)]
        stored_activations.append(('mlp_gelu', mlp_act[0].shape, 'tp'))
        
        # Step 11: Row-parallel MLP down — (b, s, 4h/N) -> (b, s, h) partial
        mlp_row = [mlp_act[r] @ self.mlp_row_weights[r] for r in range(N)]
        
        # Step 12: Reduce-Scatter — (b, s, h) -> (b, s/N, h)
        mlp_sp = ???  # YOUR CODE
        
        # Step 13: Dropout + Residual (SP layout)
        out2 = ???  # YOUR CODE
        stored_activations.append(('dropout2', out2[0].shape, 'sp'))
        stored_activations.append(('dropout2_mask', out2[0].shape, 'sp'))
        stored_activations.append(('residual', out2[0].shape, 'sp'))
        # ==============================
        
        return out2, stored_activations


# Test
H_TEST = 64
S_TEST = 32
B_TEST = 1
N_TEST = 4

dist_test = SimulatedDistributed(N_TEST)

sp_layer = TPSPLayer(H_TEST, 4, N_TEST)
sp_in = [torch.randn(B_TEST, S_TEST // N_TEST, H_TEST) for _ in range(N_TEST)]

try:
    sp_out, sp_acts = sp_layer.forward(sp_in, dist_test)
    print(f"Input (SP):  {sp_in[0].shape} per GPU")
    print(f"Output (SP): {sp_out[0].shape} per GPU")
    assert sp_out[0].shape == sp_in[0].shape, f"Shape mismatch: {sp_out[0].shape} vs {sp_in[0].shape}"
    print("TP+SP layer forward pass verified!")
except Exception as e:
    print(f"Error: {e}")
    print("Complete the TODO above.")

---


In [ ]:
#@title 🎧 Narration: Stop And Think
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_stop_and_think.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Stop and Think

1. How many AllGather operations happen in one layer? How many Reduce-Scatter?
2. What is the shape of the *largest* tensor any GPU holds at any point during the forward pass?
3. The AllGather produces a temporary full-size tensor. Why does this not defeat the purpose of SP?

---

In [ ]:
#@title 🎧 Listen: Solution Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_solution_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Solution

In [ ]:
#@title 🎧 Code Walkthrough: Tpsp Layer Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_tpsp_layer_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class TPSPLayer:
    """Transformer layer with combined Tensor + Sequence Parallelism."""
    
    def __init__(self, hidden_dim, num_heads, tp_degree, mlp_ratio=4):
        self.h = hidden_dim
        self.N = tp_degree
        self.mlp_ratio = mlp_ratio
        
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        
        self.attn_col_weights = [
            torch.randn(hidden_dim, hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.attn_row_weights = [
            torch.randn(hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
        self.mlp_col_weights = [
            torch.randn(hidden_dim, mlp_ratio * hidden_dim // tp_degree) * 0.02
            for _ in range(tp_degree)
        ]
        self.mlp_row_weights = [
            torch.randn(mlp_ratio * hidden_dim // tp_degree, hidden_dim) * 0.02
            for _ in range(tp_degree)
        ]
    
    def forward(self, sp_inputs, dist_group):
        N = self.N
        stored_activations = []
        
        # --- Attention sub-block ---
        residuals = [x.clone() for x in sp_inputs]
        
        # LayerNorm 1 — local on SP chunks
        ln1_out = [self.ln1(sp_inputs[r]) for r in range(N)]
        stored_activations.append(('ln1_input', sp_inputs[0].shape, 'sp'))
        
        # AllGather
        full_attn = dist_group.all_gather_seq(ln1_out)
        
        # Column-parallel attention
        attn_col = [full_attn[r] @ self.attn_col_weights[r] for r in range(N)]
        stored_activations.append(('attn_col_out', attn_col[0].shape, 'tp'))
        
        attn_act = [F.gelu(attn_col[r]) for r in range(N)]
        stored_activations.append(('attn_gelu', attn_act[0].shape, 'tp'))
        
        # Row-parallel
        attn_row = [attn_act[r] @ self.attn_row_weights[r] for r in range(N)]
        
        # Reduce-Scatter
        attn_sp = dist_group.reduce_scatter_seq(attn_row)
        
        # Dropout + Residual (SP)
        out1 = [F.dropout(attn_sp[r], p=0.1, training=False) + residuals[r] for r in range(N)]
        stored_activations.append(('dropout1', out1[0].shape, 'sp'))
        stored_activations.append(('dropout1_mask', out1[0].shape, 'sp'))
        
        # --- MLP sub-block ---
        residuals2 = [x.clone() for x in out1]
        
        # LayerNorm 2
        ln2_out = [self.ln2(out1[r]) for r in range(N)]
        stored_activations.append(('ln2_input', out1[0].shape, 'sp'))
        
        # AllGather
        full_mlp = dist_group.all_gather_seq(ln2_out)
        
        # Column-parallel MLP
        mlp_col = [full_mlp[r] @ self.mlp_col_weights[r] for r in range(N)]
        stored_activations.append(('mlp_col_out', mlp_col[0].shape, 'tp'))
        
        mlp_act = [F.gelu(mlp_col[r]) for r in range(N)]
        stored_activations.append(('mlp_gelu', mlp_act[0].shape, 'tp'))
        
        # Row-parallel MLP
        mlp_row = [mlp_act[r] @ self.mlp_row_weights[r] for r in range(N)]
        
        # Reduce-Scatter
        mlp_sp = dist_group.reduce_scatter_seq(mlp_row)
        
        # Dropout + Residual (SP)
        out2 = [F.dropout(mlp_sp[r], p=0.1, training=False) + residuals2[r] for r in range(N)]
        stored_activations.append(('dropout2', out2[0].shape, 'sp'))
        stored_activations.append(('dropout2_mask', out2[0].shape, 'sp'))
        stored_activations.append(('residual', out2[0].shape, 'sp'))
        
        return out2, stored_activations


# Verify
sp_layer = TPSPLayer(H_TEST, 4, N_TEST)
sp_out, sp_acts = sp_layer.forward(sp_in, dist_test)
print(f"Input (SP):  {sp_in[0].shape} per GPU")
print(f"Output (SP): {sp_out[0].shape} per GPU")
print(f"Stored activations: {len(sp_acts)}")
for name, shape, cat in sp_acts:
    print(f"  {name:<20} {str(shape):<25} {cat}")

In [ ]:
#@title 🎧 Narration: Section Memory Comparison
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_section_memory_comparison.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Side-by-Side Memory Comparison

Now let us run both layers and compare the stored activation memory.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Comparison Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_memory_comparison_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def compute_stored_memory(stored_activations, dtype_bytes=2):
    """Compute total stored activation memory from a trace."""
    total_tp = 0
    total_non_tp = 0
    total_sp = 0
    
    for name, shape, category in stored_activations:
        # Dropout masks are 1 byte, everything else is dtype_bytes
        elem_bytes = 1 if 'mask' in name else dtype_bytes
        mem = 1
        for d in shape:
            mem *= d
        mem *= elem_bytes
        
        if category == 'tp':
            total_tp += mem
        elif category == 'non-tp':
            total_non_tp += mem
        elif category == 'sp':
            total_sp += mem
    
    return {
        'tp': total_tp,
        'non_tp': total_non_tp,
        'sp': total_sp,
        'total': total_tp + total_non_tp + total_sp,
    }


# Run both at a moderately realistic scale
H_CMP = 512
S_CMP = 1024
B_CMP = 1
N_CMP = 4

dist_cmp = SimulatedDistributed(N_CMP)

# Standard TP
std_layer = StandardTPLayer(H_CMP, 8, N_CMP)
std_inputs = [torch.randn(B_CMP, S_CMP, H_CMP) for _ in range(N_CMP)]
_, std_acts = std_layer.forward(std_inputs, dist_cmp)

# TP + SP
sp_layer = TPSPLayer(H_CMP, 8, N_CMP)
sp_inputs = [torch.randn(B_CMP, S_CMP // N_CMP, H_CMP) for _ in range(N_CMP)]
_, sp_acts = sp_layer.forward(sp_inputs, dist_cmp)

std_mem = compute_stored_memory(std_acts)
sp_mem = compute_stored_memory(sp_acts)

print(f"Configuration: b={B_CMP}, s={S_CMP}, h={H_CMP}, N={N_CMP}")
print(f"")
print(f"{'Category':<25} {'Standard TP (MB)':<18} {'TP + SP (MB)':<18}")
print("-" * 62)
print(f"{'TP activations':<25} {bytes_to_mb(std_mem['tp']):<18.2f} {bytes_to_mb(sp_mem['tp']):<18.2f}")
print(f"{'Non-TP (replicated)':<25} {bytes_to_mb(std_mem['non_tp']):<18.2f} {'—':<18}")
print(f"{'SP (distributed)':<25} {'—':<18} {bytes_to_mb(sp_mem['sp']):<18.2f}")
print(f"{'-'*62}")
print(f"{'Total per GPU':<25} {bytes_to_mb(std_mem['total']):<18.2f} {bytes_to_mb(sp_mem['total']):<18.2f}")
print(f"")
print(f"Memory saved per GPU: {bytes_to_mb(std_mem['total'] - sp_mem['total']):.2f} MB")
print(f"Reduction: {std_mem['total'] / sp_mem['total']:.2f}x")

In [ ]:
#@title 🎧 Narration: Section Communication Cost
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_section_communication_cost.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Communication Cost Verification

Let us verify that TP+SP uses exactly the same communication bandwidth as standard TP.

In [ ]:
#@title 🎧 Code Walkthrough: Communication Cost Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_communication_cost_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Track communication for both approaches

# Standard TP
dist_std = SimulatedDistributed(N_CMP)
dist_std.reset_comm_counter()
std_layer.forward(std_inputs, dist_std)
std_comm = dist_std.comm_bytes

# TP + SP
dist_sp = SimulatedDistributed(N_CMP)
dist_sp.reset_comm_counter()
sp_layer.forward(sp_inputs, dist_sp)
sp_comm = dist_sp.comm_bytes

print(f"Communication per layer (b={B_CMP}, s={S_CMP}, h={H_CMP}, N={N_CMP}):")
print(f"  Standard TP (2 AllReduces):                {bytes_to_mb(std_comm):.2f} MB")
print(f"  TP + SP (2 AllGather + 2 Reduce-Scatter):  {bytes_to_mb(sp_comm):.2f} MB")
print(f"  Ratio: {sp_comm / std_comm:.4f}")
print(f"\nCommunication is identical (within floating point) — zero extra cost!")

In [ ]:
#@title 🎧 Narration: Section Scaling Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_section_scaling_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Scaling Analysis: Memory vs Sequence Length and TP Degree

Let us see how the memory savings scale with different configurations.

In [ ]:
#@title 🎧 Code Walkthrough: Analytical Memory Function
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_analytical_memory_function.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analytical_memory(h, s, b, N, num_layers, dtype_bytes=2):
    """Compute analytical activation memory for standard TP and TP+SP."""
    
    # Per layer stored activations:
    # TP activations (same in both): attn_col, attn_gelu, mlp_col, mlp_gelu
    tp_per_layer = (
        2 * b * s * (h // N) * dtype_bytes +  # attn col + gelu
        2 * b * s * (4 * h // N) * dtype_bytes  # mlp col + gelu
    )
    
    # Non-TP activations:
    # Standard TP: 5 full tensors + 2 masks
    non_tp_std = 5 * b * s * h * dtype_bytes + 2 * b * s * h * 1
    
    # SP activations:
    # Same count but at (b, s/N, h)
    sp_per_layer = 5 * b * (s // N) * h * dtype_bytes + 2 * b * (s // N) * h * 1
    
    std_total = (tp_per_layer + non_tp_std) * num_layers
    sp_total = (tp_per_layer + sp_per_layer) * num_layers
    
    return {
        'std_gb': std_total / 1e9,
        'sp_gb': sp_total / 1e9,
        'saved_gb': (std_total - sp_total) / 1e9,
    }


# Vary sequence length
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: varying sequence length (h=8192, N=8, L=80)
ax = axes[0]
seq_lens = [512, 1024, 2048, 4096, 8192, 16384]
h_fixed, N_fixed, L_fixed = 8192, 8, 80

std_mems = [analytical_memory(h_fixed, s, 1, N_fixed, L_fixed)['std_gb'] for s in seq_lens]
sp_mems = [analytical_memory(h_fixed, s, 1, N_fixed, L_fixed)['sp_gb'] for s in seq_lens]

ax.plot(seq_lens, std_mems, 'o-', color='#F44336', linewidth=2, markersize=8, label='Standard TP')
ax.plot(seq_lens, sp_mems, 's-', color='#4CAF50', linewidth=2, markersize=8, label='TP + SP')
ax.fill_between(seq_lens, sp_mems, std_mems, alpha=0.15, color='green')

ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Total Activation Memory per GPU (GB)', fontsize=12)
ax.set_title(f'Memory vs Sequence Length\n(h={h_fixed}, N={N_fixed}, L={L_fixed})', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_xscale('log', base=2)

# Right: varying TP degree (h=8192, s=2048, L=80)
ax = axes[1]
tp_degrees = [1, 2, 4, 8, 16]
s_fixed = 2048

std_mems_tp = [analytical_memory(h_fixed, s_fixed, 1, n, L_fixed)['std_gb'] for n in tp_degrees]
sp_mems_tp = [analytical_memory(h_fixed, s_fixed, 1, n, L_fixed)['sp_gb'] for n in tp_degrees]
savings_tp = [analytical_memory(h_fixed, s_fixed, 1, n, L_fixed)['saved_gb'] for n in tp_degrees]

x_pos = np.arange(len(tp_degrees))
width = 0.35

ax.bar(x_pos - width/2, std_mems_tp, width, label='Standard TP', color='#F44336', alpha=0.85)
ax.bar(x_pos + width/2, sp_mems_tp, width, label='TP + SP', color='#4CAF50', alpha=0.85)

# Add savings labels
for i, (s_val, sp_val) in enumerate(zip(std_mems_tp, sp_mems_tp)):
    if s_val > sp_val:
        ax.annotate(f'-{s_val - sp_val:.1f} GB',
                    xy=(i, max(s_val, sp_val) + 0.5),
                    ha='center', fontsize=9, color='green', fontweight='bold')

ax.set_xlabel('TP Degree (N)', fontsize=12)
ax.set_ylabel('Total Activation Memory per GPU (GB)', fontsize=12)
ax.set_title(f'Memory vs TP Degree\n(h={h_fixed}, s={s_fixed}, L={L_fixed})', fontsize=13)
ax.set_xticks(x_pos)
ax.set_xticklabels(tp_degrees)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
#@title 🎧 What to Look For: Scaling Analysis Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_scaling_analysis_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("Key observations:")
print("1. The gap between TP and TP+SP grows with sequence length")
print("2. Higher TP degree means more memory saved by SP")
print("3. SP is always beneficial — there is no configuration where it hurts")

In [ ]:
#@title 🎧 Narration: Section Gpu Memory Profiling
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_section_gpu_memory_profiling.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. GPU Memory Profiling (Live Measurement)

Let us do actual GPU memory measurements by running forward passes with PyTorch tensors on GPU.

In [ ]:
#@title 🎧 Code Walkthrough: Measure Activation Memory Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_measure_activation_memory_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def measure_activation_memory(hidden_dim, seq_len, tp_degree, use_sp=False):
    """Measure actual GPU memory for stored activations.
    
    Simulates storing the activations that would be saved for backward pass.
    """
    b = 1
    h = hidden_dim
    s = seq_len
    N = tp_degree
    
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        baseline = torch.cuda.memory_allocated()
    
    stored = []
    
    if use_sp:
        # SP stored activations: (b, s/N, h) for non-TP ops
        # LN1 input, LN2 input, dropout1, dropout2, residual
        for _ in range(5):
            stored.append(torch.randn(b, s // N, h, dtype=torch.bfloat16, device=dev))
        # Dropout masks
        for _ in range(2):
            stored.append(torch.randint(0, 2, (b, s // N, h), dtype=torch.uint8, device=dev))
    else:
        # Standard TP stored activations: (b, s, h) for non-TP ops
        for _ in range(5):
            stored.append(torch.randn(b, s, h, dtype=torch.bfloat16, device=dev))
        for _ in range(2):
            stored.append(torch.randint(0, 2, (b, s, h), dtype=torch.uint8, device=dev))
    
    # TP stored activations (same in both): (b, s, h/N) and (b, s, 4h/N)
    for _ in range(2):  # attn col + gelu
        stored.append(torch.randn(b, s, h // N, dtype=torch.bfloat16, device=dev))
    for _ in range(2):  # mlp col + gelu
        stored.append(torch.randn(b, s, 4 * h // N, dtype=torch.bfloat16, device=dev))
    
    if torch.cuda.is_available():
        mem_used = torch.cuda.memory_allocated() - baseline
    else:
        # Estimate on CPU
        mem_used = sum(t.numel() * t.element_size() for t in stored)
    
    # Clean up
    del stored
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return mem_used


# Measure at different scales
configs_measure = [
    (256,  512,  4),
    (512,  1024, 4),
    (1024, 2048, 4),
    (1024, 2048, 8),
]

print(f"{'Config (h, s, N)':<25} {'Standard TP (MB)':<18} {'TP + SP (MB)':<18} {'Savings (MB)':<15} {'Factor':<8}")
print("-" * 85)

for h, s, N in configs_measure:
    std_mem = measure_activation_memory(h, s, N, use_sp=False)
    sp_mem = measure_activation_memory(h, s, N, use_sp=True)
    
    std_mb = bytes_to_mb(std_mem)
    sp_mb = bytes_to_mb(sp_mem)
    saved_mb = std_mb - sp_mb
    factor = std_mem / sp_mem if sp_mem > 0 else float('inf')
    
    config_str = f"h={h}, s={s}, N={N}"
    print(f"{config_str:<25} {std_mb:<18.2f} {sp_mb:<18.2f} {saved_mb:<15.2f} {factor:<8.2f}x")

In [ ]:
#@title 🎧 Narration: Section Impact On Real Models
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_section_impact_on_real_models.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Impact on Real Models: End-to-End Memory Budget

Let us compute the full memory budget for training a large model, showing how SP impacts the total memory.

In [ ]:
#@title 🎧 Code Walkthrough: Full Memory Budget Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_full_memory_budget_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def full_memory_budget(model_name, params_b, hidden_dim, num_layers, num_heads,
                       seq_len, batch_size, tp_degree, dtype_bytes=2):
    """Compute full training memory budget including params, grads, optimizer, and activations."""
    h, L, s, b, N = hidden_dim, num_layers, seq_len, batch_size, tp_degree
    
    # Parameters per GPU (split by TP)
    total_params = params_b * 1e9
    params_per_gpu = total_params / N
    
    # Memory components (in bytes):
    # Parameters (BF16)
    param_mem = params_per_gpu * dtype_bytes
    
    # Gradients (BF16)
    grad_mem = params_per_gpu * dtype_bytes
    
    # Optimizer states (Adam: 2 states in FP32 + master weights in FP32)
    optimizer_mem = params_per_gpu * (4 + 4 + 4)  # momentum, variance, master weights
    
    # Activation memory per GPU per layer
    # TP activations (same either way)
    tp_act = (2 * b * s * (h // N) + 2 * b * s * (4 * h // N)) * dtype_bytes
    
    # Non-TP activations
    non_tp_std = (5 * b * s * h * dtype_bytes) + (2 * b * s * h * 1)
    non_tp_sp = (5 * b * (s // N) * h * dtype_bytes) + (2 * b * (s // N) * h * 1)
    
    act_std_total = (tp_act + non_tp_std) * L
    act_sp_total = (tp_act + non_tp_sp) * L
    
    total_std = param_mem + grad_mem + optimizer_mem + act_std_total
    total_sp = param_mem + grad_mem + optimizer_mem + act_sp_total
    
    return {
        'params_gb': param_mem / 1e9,
        'grads_gb': grad_mem / 1e9,
        'optimizer_gb': optimizer_mem / 1e9,
        'act_std_gb': act_std_total / 1e9,
        'act_sp_gb': act_sp_total / 1e9,
        'total_std_gb': total_std / 1e9,
        'total_sp_gb': total_sp / 1e9,
        'act_saved_gb': (act_std_total - act_sp_total) / 1e9,
    }


models = [
    ('LLaMA 7B',   7,   4096,  32, 32),
    ('LLaMA 70B',  70,  8192,  80, 64),
    ('GPT-3 175B', 175, 12288, 96, 96),
]

print(f"Training memory budget per GPU (s=2048, b=1, BF16, Adam, N=8)")
print(f"")

for name, params, h, L, heads in models:
    budget = full_memory_budget(name, params, h, L, heads, 2048, 1, 8)
    
    print(f"--- {name} ---")
    print(f"  Parameters:     {budget['params_gb']:.1f} GB")
    print(f"  Gradients:      {budget['grads_gb']:.1f} GB")
    print(f"  Optimizer:      {budget['optimizer_gb']:.1f} GB")
    print(f"  Activations:")
    print(f"    Standard TP:  {budget['act_std_gb']:.1f} GB")
    print(f"    TP + SP:      {budget['act_sp_gb']:.1f} GB  (saved {budget['act_saved_gb']:.1f} GB)")
    print(f"  Total:")
    print(f"    Standard TP:  {budget['total_std_gb']:.1f} GB")
    print(f"    TP + SP:      {budget['total_sp_gb']:.1f} GB")
    print(f"  A100 80GB fit?  {'Yes' if budget['total_sp_gb'] < 80 else 'No'} (with SP) / {'Yes' if budget['total_std_gb'] < 80 else 'No'} (without SP)")
    print()

In [ ]:
#@title 🎧 What to Look For: Memory Budget Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_memory_budget_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualization: stacked bar chart of memory budget

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, params, h, L, heads) in enumerate(models):
    budget = full_memory_budget(name, params, h, L, heads, 2048, 1, 8)
    ax = axes[idx]
    
    categories = ['Params', 'Gradients', 'Optimizer', 'Activations']
    std_vals = [budget['params_gb'], budget['grads_gb'], budget['optimizer_gb'], budget['act_std_gb']]
    sp_vals = [budget['params_gb'], budget['grads_gb'], budget['optimizer_gb'], budget['act_sp_gb']]
    
    colors = ['#2196F3', '#FF9800', '#9C27B0', '#F44336']
    sp_colors = ['#2196F3', '#FF9800', '#9C27B0', '#4CAF50']
    
    x = [0, 1]
    width = 0.6
    
    # Stacked bars
    bottom_std = 0
    bottom_sp = 0
    for i, cat in enumerate(categories):
        ax.bar(0, std_vals[i], width, bottom=bottom_std, color=colors[i],
               alpha=0.85, label=cat if idx == 0 else None)
        label_sp = f'{cat} (SP)' if cat == 'Activations' else None
        ax.bar(1, sp_vals[i], width, bottom=bottom_sp, color=sp_colors[i],
               alpha=0.85, label=label_sp if idx == 0 else None)
        bottom_std += std_vals[i]
        bottom_sp += sp_vals[i]
    
    # A100 80GB line
    ax.axhline(y=80, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
    ax.text(0.5, 82, '80 GB (A100)', ha='center', fontsize=9, color='red')
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Standard TP', 'TP + SP'])
    ax.set_ylabel('Memory per GPU (GB)', fontsize=11)
    ax.set_title(f'{name}\n(N=8, s=2048)', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    # Total labels
    ax.text(0, bottom_std + 1, f'{budget["total_std_gb"]:.0f} GB',
            ha='center', fontsize=10, fontweight='bold')
    ax.text(1, bottom_sp + 1, f'{budget["total_sp_gb"]:.0f} GB',
            ha='center', fontsize=10, fontweight='bold', color='green')

axes[0].legend(fontsize=9, loc='upper left')

plt.suptitle('Training Memory Budget: Standard TP vs TP + Sequence Parallelism',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Narration: Section Practical Considerations
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_section_practical_considerations.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Practical Considerations

Let us verify the key practical constraints mentioned in the article.

In [ ]:
#@title 🎧 Code Walkthrough: Constraint Seq Len Divisible
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_constraint_seq_len_divisible.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Constraint 1: Sequence length must be divisible by TP degree

print("=== Constraint: seq_len must be divisible by N ===")
print("")

test_cases = [
    (2048, 4, True),
    (2048, 8, True),
    (4096, 4, True),
    (1000, 4, False),  # 1000 / 4 = 250 (works but unusual)
    (2048, 3, False),  # 2048 / 3 = 682.67 (does NOT work)
]

for s, N, expected_good in test_cases:
    divisible = s % N == 0
    chunk_size = s / N
    status = "OK" if divisible else "FAIL"
    print(f"  s={s}, N={N}: s/N = {chunk_size:.2f}  [{status}]")

print(f"\nPractice: Training sequences are powers of 2 (1024, 2048, 4096)")
print(f"         TP degrees are small powers of 2 (2, 4, 8)")
print(f"         So this constraint is rarely an issue.")

In [ ]:
#@title 🎧 Narration: Constraint Dropout Masks
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_constraint_dropout_masks.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Constraint 2: Dropout masks must be consistent between forward and backward

print("=== Constraint: Dropout mask consistency ===")
print("")

# In SP, each GPU generates its own dropout mask for its s/N tokens
# This is naturally consistent because each GPU uses its own random state

torch.manual_seed(42)
s, h, N = 16, 8, 4

# Simulate: GPU 0 generates mask for tokens 0-3
# GPU 1 generates mask for tokens 4-7, etc.
masks = []
for rank in range(N):
    # Each GPU has its own random seed in practice
    torch.manual_seed(42 + rank)
    x = torch.ones(1, s // N, h)
    mask = F.dropout(x, p=0.5, training=True)
    masks.append(mask)
    kept = (mask != 0).sum().item()
    total = mask.numel()
    print(f"  GPU {rank}: {kept}/{total} elements kept ({kept/total*100:.0f}%)")

# Verify: running again with same seed gives same mask
print(f"")
print(f"  Verification: same seed -> same mask")
for rank in range(N):
    torch.manual_seed(42 + rank)
    x = torch.ones(1, s // N, h)
    mask2 = F.dropout(x, p=0.5, training=True)
    same = torch.allclose(masks[rank], mask2)
    print(f"  GPU {rank}: masks identical? {same}")

print(f"\n  Each GPU's mask is independent and reproducible.")
print(f"  No cross-GPU coordination needed for dropout.")

In [ ]:
#@title 🎧 Narration: Section Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_section_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 10. Summary and Complete Picture

Let us summarize everything we have learned across the three notebooks with a final comprehensive comparison.

In [ ]:
# Final summary table


In [ ]:
#@title 🎧 Listen: Summary Table
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_summary_table.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
print("=" * 75)
print("SEQUENCE PARALLELISM: COMPLETE SUMMARY")
print("=" * 75)
print("")
print("The Problem (Notebook 1):")
print("  - LayerNorm needs full hidden dim -> cannot split along h")
print("  - Standard TP replicates non-TP activations on every GPU")
print("  - Non-TP activations ~ 30% of total activation memory")
print("  - For 175B model, TP=8: ~20 GB wasted per GPU")
print("")
print("The Solution (Notebook 2):")
print("  - LayerNorm is independent per token -> CAN split along sequence")
print("  - AllGather: (b, s/N, h) -> (b, s, h) before TP regions")
print("  - Reduce-Scatter: (b, s, h) -> (b, s/N, h) after TP regions")
print("  - AllReduce = Reduce-Scatter + AllGather (same bandwidth)")
print("  - ZERO extra communication cost")
print("")
print("The Result (Notebook 3):")
print("  - Non-TP activation memory reduced by factor of N")
print("  - Communication cost unchanged")
print("  - Always beneficial, no configuration where it hurts")
print("")
print("+" + "-" * 73 + "+")
print(f"| {'Property':<35} | {'Standard TP':<15} | {'TP + SP':<15} |")
print("+" + "-" * 73 + "+")
print(f"| {'LayerNorm activations per GPU':<35} | {'(b, s, h)':<15} | {'(b, s/N, h)':<15} |")
print(f"| {'Dropout activations per GPU':<35} | {'(b, s, h)':<15} | {'(b, s/N, h)':<15} |")
print(f"| {'MLP/Attention activations per GPU':<35} | {'(b, s, h/N)':<15} | {'(b, s, h/N)':<15} |")
print(f"| {'Communication per sub-block':<35} | {'1 AllReduce':<15} | {'1 AG + 1 RS':<15} |")
print(f"| {'Total comm bandwidth per layer':<35} | {'2 AllReduce':<15} | {'2 AG + 2 RS':<15} |")
print(f"| {'Bandwidth equivalence':<35} | {'2(N-1)/N * M':<15} | {'2(N-1)/N * M':<15} |")
print(f"| {'Non-TP memory reduction':<35} | {'1x (none)':<15} | {'Nx':<15} |")
print("+" + "-" * 73 + "+")
print("")
print("Next: Context Parallelism — splitting the sequence for ATTENTION")
print("(not just LayerNorm/dropout), enabling million-token contexts.")

In [ ]:
#@title 🎧 Narration: Section Reflection Challenges
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_section_reflection_challenges.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 11. Reflection and Optional Challenges

### Reflection Questions

1. **Fused kernels**: The article mentions that in practice, AllGather is fused with the matrix multiplication so the full tensor is never fully materialized. Why is this important? What would happen to peak memory if we materialized it?

2. **Backward pass**: In the backward pass, the roles of AllGather and Reduce-Scatter are swapped. Can you trace through the backward pass and verify this?

3. **Context parallelism**: We split the sequence for operations that are independent per token (LayerNorm, dropout). But attention has cross-token interactions ($QK^T$). Why can we not use the same approach for attention? What additional communication would be needed?

### Optional Challenges

1. **Implement the backward pass**: Add backward pass tracing to the `TPSPLayer` class and verify that the communication volume is identical.

2. **Overlap communication and compute**: Modify the forward pass to simulate overlapping the AllGather with the column-parallel linear layer (process chunks as they arrive).

3. **Activation checkpointing analysis**: If we add activation checkpointing (recompute instead of store), how does it interact with SP? Does SP still provide savings?